# CP2 baselines on Kaggle: S0-noLegal, S0, S1 (Qwen3-4B) and B0

**Notebook settings (right sidebar → Session options):** Accelerator **GPU T4 x2** (one GPU is used),
**Internet on**. If the GitHub repo is private, add a Kaggle secret **`GITHUB_TOKEN`** (Add-ons → Secrets) with
read access to `Zografska/greenwashing-apprentice`.

Order: run everything on **dev** first, look at the numbers, and only then run the **test** cells once
(`--final`). Download `results_baselines.zip` at the end, unzip it into the repo and commit.
GPU quota: roughly 15–30 minutes in total. Stop the session when done.

In [ ]:
import subprocess, os
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = (f"https://{token}@github.com/Zografska/greenwashing-apprentice.git" if token
       else "https://github.com/Zografska/greenwashing-apprentice.git")
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--depth", "1", url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Qwen3 needs transformers >= 4.51; the rest is light.
!pip install -q -U "transformers>=4.51" sentence-transformers scikit-learn scipy openpyxl pytest
import torch, transformers, sentence_transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| sentence-transformers", sentence_transformers.__version__, "| GPU", torch.cuda.get_device_name(0))

In [ ]:
# Offline sanity checks: split frozen and untouched, prompts leak-free
!python -m pytest -q tests
!python -m apprentice.check_split | tail -3

## Dev runs

In [ ]:
!python -m apprentice.baselines prompted --condition S0-noLegal --suite dev --append
!python -m apprentice.baselines prompted --condition S0 --suite dev --append
!python -m apprentice.baselines prompted --condition S1 --suite dev --append

In [ ]:
!python -m apprentice.baselines b0 --labels teacher --suite dev --append
!python -m apprentice.baselines b0 --labels gold --suite dev --append

## Test runs (once)

Run these only after the dev numbers look sane (valid predictions, no crash, ms/claim reported). They write
`results/preds/<condition>/test.jsonl` and refuse to overwrite it.

In [ ]:
!python -m apprentice.baselines prompted --condition S0-noLegal --suite test --final --append
!python -m apprentice.baselines prompted --condition S0 --suite test --final --append
!python -m apprentice.baselines prompted --condition S1 --suite test --final --append
!python -m apprentice.baselines b0 --labels teacher --suite test --final --append
!python -m apprentice.baselines b0 --labels gold --suite test --final --append

## Download the results

In [ ]:
!zip -qr /kaggle/working/results_baselines.zip results/preds results/results.csv
print("Download /kaggle/working/results_baselines.zip from the Output panel, unzip it in the repo root, commit.")